In [ ]:
# Cell 1 - Install/Import Libraries
# Run this cell once. If PyTorch is already installed, the install command is skipped.

import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch", "--quiet"])

import os
import time
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")


In [ ]:
# Cell 2 - Load and Validate Dataset

file_name = "RELIANCE_2015_2025_Daily.csv.csv"

if not os.path.exists(file_name):
    raise FileNotFoundError(
        f"Dataset not found: {file_name}\n"
        f"Current folder: {os.getcwd()}\n"
        f"Files found: {os.listdir()}"
    )

df = pd.read_csv(file_name)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

print("Dataset shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nFirst 5 rows:")
print(df.head())
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nDate range:", df["Date"].min().date(), "to", df["Date"].max().date())


FileNotFoundError: Dataset not found: RELIANCE_2015_2025_Daily.csv.csv
Current folder: c:\Users\myesh\Downloads
Files found: ['Assignment_2___DAV.zip', 'cloudlens_pipeline_stage1_ingestion.png', 'cloudlens_pipeline_stage2_diagnosis.png', 'CSE-Rhinos  PBL Excel.xlsx', 'DL_Internal_Assessments_Projects_Others.pdf', 'Stock_Price_Forecasting_RNN_LSTM_GRU_CORRECTED.ipynb']

In [ ]:
# Cell 3 - EDA: Closing Price

plt.figure(figsize=(12, 6))
plt.plot(df["Date"], df["Close"])
plt.title("Reliance Industries Closing Price (2015-2025)")
plt.xlabel("Date")
plt.ylabel("Closing Price")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 4 - EDA: Trading Volume

plt.figure(figsize=(12, 6))
plt.plot(df["Date"], df["Volume"])
plt.title("Reliance Industries Trading Volume (2015-2025)")
plt.xlabel("Date")
plt.ylabel("Trading Volume")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 5 - EDA: OHLC Prices

plt.figure(figsize=(12, 6))
plt.plot(df["Date"], df["Open"], label="Open")
plt.plot(df["Date"], df["High"], label="High")
plt.plot(df["Date"], df["Low"], label="Low")
plt.plot(df["Date"], df["Close"], label="Close")
plt.title("Reliance Industries OHLC Prices (2015-2025)")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 6 - EDA: Closing Price Distribution

plt.figure(figsize=(8, 5))
plt.hist(df["Close"], bins=30)
plt.title("Distribution of Reliance Closing Prices")
plt.xlabel("Closing Price")
plt.ylabel("Frequency")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 7 - Select Target

data = df[["Date", "Close"]].copy()

print(data.head())
print("\nShape:", data.shape)


In [ ]:
# Cell 8 - Chronological 80/10/10 Split

train_size = int(len(data) * 0.80)
val_size = int(len(data) * 0.10)

train_data = data.iloc[:train_size].copy()
val_data = data.iloc[train_size:train_size + val_size].copy()
test_data = data.iloc[train_size + val_size:].copy()

print("Training set:", train_data.shape)
print("Validation set:", val_data.shape)
print("Testing set:", test_data.shape)

print("\nTraining:", train_data["Date"].min().date(), "to", train_data["Date"].max().date())
print("Validation:", val_data["Date"].min().date(), "to", val_data["Date"].max().date())
print("Testing:", test_data["Date"].min().date(), "to", test_data["Date"].max().date())


In [ ]:
# Cell 9 - Min-Max Scaling Without Leakage

scaler = MinMaxScaler(feature_range=(0, 1))

train_scaled = scaler.fit_transform(train_data[["Close"]])
val_scaled = scaler.transform(val_data[["Close"]])
test_scaled = scaler.transform(test_data[["Close"]])

print("Train scaled:", train_scaled.shape)
print("Validation scaled:", val_scaled.shape)
print("Test scaled:", test_scaled.shape)
print("\nFirst 5 scaled training values:")
print(train_scaled[:5])


In [ ]:
# Cell 10 - Sequence Creation

sequence_length = 60

def create_sequences(values, sequence_length):
    X, y = [], []

    for i in range(sequence_length, len(values)):
        X.append(values[i-sequence_length:i, 0])
        y.append(values[i, 0])

    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32)

# Training sequences
X_train, y_train = create_sequences(train_scaled, sequence_length)

# IMPORTANT:
# Validation and test sequences include the immediately preceding observations
# as context. This prevents losing the first 60 validation/test targets.

val_context = np.vstack([train_scaled[-sequence_length:], val_scaled])
test_context = np.vstack([val_scaled[-sequence_length:], test_scaled])

X_val, y_val = create_sequences(val_context, sequence_length)
X_test, y_test = create_sequences(test_context, sequence_length)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)


In [ ]:
# Cell 11 - Convert to PyTorch Tensors

# Shape required by recurrent layers:
# (samples, time_steps, features)

X_train_tensor = torch.tensor(X_train).unsqueeze(-1)
y_train_tensor = torch.tensor(y_train).unsqueeze(-1)

X_val_tensor = torch.tensor(X_val).unsqueeze(-1)
y_val_tensor = torch.tensor(y_val).unsqueeze(-1)

X_test_tensor = torch.tensor(X_test).unsqueeze(-1)
y_test_tensor = torch.tensor(y_test).unsqueeze(-1)

print("X_train:", X_train_tensor.shape)
print("y_train:", y_train_tensor.shape)
print("X_val:", X_val_tensor.shape)
print("y_val:", y_val_tensor.shape)
print("X_test:", X_test_tensor.shape)
print("y_test:", y_test_tensor.shape)


In [ ]:
# Cell 12 - DataLoaders

batch_size = 32

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# No shuffling: the data remains in chronological order.
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))


In [ ]:
# Cell 13 - Model Definitions

class VanillaRNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        output, _ = self.rnn(x)
        return self.fc(output[:, -1, :])


class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        output, _ = self.lstm(x)
        return self.fc(output[:, -1, :])


class GRUModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, num_layers=1):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        output, _ = self.gru(x)
        return self.fc(output[:, -1, :])


In [ ]:
# Cell 14 - Training Function With Gradient-Norm Tracking

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_model(model, train_loader, val_loader, epochs=30, learning_rate=0.001):
    model = model.to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    train_losses = []
    val_losses = []
    gradient_norms = []

    start_time = time.time()

    for epoch in range(epochs):

        model.train()
        running_train_loss = 0.0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)

            loss.backward()

            # Global L2 gradient norm
            total_norm_sq = 0.0
            for parameter in model.parameters():
                if parameter.grad is not None:
                    param_norm = parameter.grad.detach().norm(2).item()
                    total_norm_sq += param_norm ** 2

            gradient_norms.append(total_norm_sq ** 0.5)

            optimizer.step()

            running_train_loss += loss.item()

        train_loss = running_train_loss / len(train_loader)

        model.eval()
        running_val_loss = 0.0

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                predictions = model(X_batch)
                loss = criterion(predictions, y_batch)

                running_val_loss += loss.item()

        val_loss = running_val_loss / len(val_loader)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(
                f"Epoch {epoch+1:02d}/{epochs} | "
                f"Train Loss: {train_loss:.6f} | "
                f"Val Loss: {val_loss:.6f}"
            )

    training_time = time.time() - start_time

    return model, train_losses, val_losses, gradient_norms, training_time


In [ ]:
# Cell 15 - Evaluation Function

def predict_model(model, data_loader):
    model.eval()

    predictions = []
    actual = []

    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch = X_batch.to(device)

            output = model(X_batch)

            predictions.extend(output.cpu().numpy().ravel())
            actual.extend(y_batch.numpy().ravel())

    return np.asarray(actual), np.asarray(predictions)


def evaluate_model(model, data_loader, scaler):
    actual_scaled, predicted_scaled = predict_model(model, data_loader)

    actual = scaler.inverse_transform(
        actual_scaled.reshape(-1, 1)
    ).ravel()

    predicted = scaler.inverse_transform(
        predicted_scaled.reshape(-1, 1)
    ).ravel()

    mse = mean_squared_error(actual, predicted)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(actual, predicted)
    r2 = r2_score(actual, predicted)

    return {
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "Actual": actual,
        "Predicted": predicted
    }


In [ ]:
# Cell 16 - Train Vanilla RNN

rnn_model = VanillaRNN(input_size=1, hidden_size=50, num_layers=1)

rnn_model, rnn_train_losses, rnn_val_losses, rnn_gradients, rnn_time = train_model(
    rnn_model,
    train_loader,
    val_loader,
    epochs=30,
    learning_rate=0.001
)

rnn_results = evaluate_model(rnn_model, test_loader, scaler)

print("\nVanilla RNN Test Results")
print("------------------------")
print("MSE :", rnn_results["MSE"])
print("RMSE:", rnn_results["RMSE"])
print("MAE :", rnn_results["MAE"])
print("R²  :", rnn_results["R2"])
print("Training time:", round(rnn_time, 2), "seconds")


In [ ]:
# Cell 17 - Vanilla RNN Loss

plt.figure(figsize=(10, 5))
plt.plot(rnn_train_losses, label="Training Loss")
plt.plot(rnn_val_losses, label="Validation Loss")
plt.title("Vanilla RNN Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 18 - Vanilla RNN Predictions

plt.figure(figsize=(12, 6))
plt.plot(rnn_results["Actual"], label="Actual")
plt.plot(rnn_results["Predicted"], label="Predicted")
plt.title("Vanilla RNN - Actual vs Predicted Closing Price")
plt.xlabel("Test Time Step")
plt.ylabel("Closing Price")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 19 - Train LSTM

lstm_model = LSTMModel(input_size=1, hidden_size=50, num_layers=1)

lstm_model, lstm_train_losses, lstm_val_losses, lstm_gradients, lstm_time = train_model(
    lstm_model,
    train_loader,
    val_loader,
    epochs=30,
    learning_rate=0.001
)

lstm_results = evaluate_model(lstm_model, test_loader, scaler)

print("\nLSTM Test Results")
print("-----------------")
print("MSE :", lstm_results["MSE"])
print("RMSE:", lstm_results["RMSE"])
print("MAE :", lstm_results["MAE"])
print("R²  :", lstm_results["R2"])
print("Training time:", round(lstm_time, 2), "seconds")


In [ ]:
# Cell 20 - LSTM Loss

plt.figure(figsize=(10, 5))
plt.plot(lstm_train_losses, label="Training Loss")
plt.plot(lstm_val_losses, label="Validation Loss")
plt.title("LSTM Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 21 - LSTM Predictions

plt.figure(figsize=(12, 6))
plt.plot(lstm_results["Actual"], label="Actual")
plt.plot(lstm_results["Predicted"], label="Predicted")
plt.title("LSTM - Actual vs Predicted Closing Price")
plt.xlabel("Test Time Step")
plt.ylabel("Closing Price")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 22 - Train GRU

gru_model = GRUModel(input_size=1, hidden_size=50, num_layers=1)

gru_model, gru_train_losses, gru_val_losses, gru_gradients, gru_time = train_model(
    gru_model,
    train_loader,
    val_loader,
    epochs=30,
    learning_rate=0.001
)

gru_results = evaluate_model(gru_model, test_loader, scaler)

print("\nGRU Test Results")
print("----------------")
print("MSE :", gru_results["MSE"])
print("RMSE:", gru_results["RMSE"])
print("MAE :", gru_results["MAE"])
print("R²  :", gru_results["R2"])
print("Training time:", round(gru_time, 2), "seconds")


In [ ]:
# Cell 23 - GRU Loss

plt.figure(figsize=(10, 5))
plt.plot(gru_train_losses, label="Training Loss")
plt.plot(gru_val_losses, label="Validation Loss")
plt.title("GRU Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 24 - GRU Predictions

plt.figure(figsize=(12, 6))
plt.plot(gru_results["Actual"], label="Actual")
plt.plot(gru_results["Predicted"], label="Predicted")
plt.title("GRU - Actual vs Predicted Closing Price")
plt.xlabel("Test Time Step")
plt.ylabel("Closing Price")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 25 - RNN vs LSTM vs GRU Results

results = pd.DataFrame({
    "Model": ["Vanilla RNN", "LSTM", "GRU"],
    "MSE": [
        rnn_results["MSE"],
        lstm_results["MSE"],
        gru_results["MSE"]
    ],
    "RMSE": [
        rnn_results["RMSE"],
        lstm_results["RMSE"],
        gru_results["RMSE"]
    ],
    "MAE": [
        rnn_results["MAE"],
        lstm_results["MAE"],
        gru_results["MAE"]
    ],
    "R2": [
        rnn_results["R2"],
        lstm_results["R2"],
        gru_results["R2"]
    ],
    "Training Time (sec)": [
        rnn_time,
        lstm_time,
        gru_time
    ]
})

results


In [ ]:
# Cell 26 - RMSE Comparison

plt.figure(figsize=(9, 5))
plt.bar(results["Model"], results["RMSE"])
plt.title("RNN vs LSTM vs GRU - RMSE Comparison")
plt.xlabel("Model")
plt.ylabel("RMSE")
plt.grid(axis="y")
plt.tight_layout()
plt.show()


In [ ]:
# Cell 27 - All Predictions Comparison

plt.figure(figsize=(12, 6))

plt.plot(rnn_results["Actual"], label="Actual")
plt.plot(rnn_results["Predicted"], label="Vanilla RNN")
plt.plot(lstm_results["Predicted"], label="LSTM")
plt.plot(gru_results["Predicted"], label="GRU")

plt.title("Comparison of RNN, LSTM and GRU Predictions")
plt.xlabel("Test Time Step")
plt.ylabel("Closing Price")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 28 - Gradient Norm Comparison

plt.figure(figsize=(10, 5))

plt.plot(rnn_gradients, label="Vanilla RNN")
plt.plot(lstm_gradients, label="LSTM")
plt.plot(gru_gradients, label="GRU")

plt.title("Gradient Norm Comparison")
plt.xlabel("Training Batch")
plt.ylabel("Global Gradient Norm")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print("Average RNN gradient norm :", np.mean(rnn_gradients))
print("Average LSTM gradient norm:", np.mean(lstm_gradients))
print("Average GRU gradient norm :", np.mean(gru_gradients))


In [ ]:
# Cell 29 - Sequence Length Experiment

def make_loaders_for_sequence_length(seq_len):
    X_tr, y_tr = create_sequences(train_scaled, seq_len)

    val_ctx = np.vstack([train_scaled[-seq_len:], val_scaled])
    X_v, y_v = create_sequences(val_ctx, seq_len)

    X_tr = torch.tensor(X_tr).unsqueeze(-1)
    y_tr = torch.tensor(y_tr).unsqueeze(-1)
    X_v = torch.tensor(X_v).unsqueeze(-1)
    y_v = torch.tensor(y_v).unsqueeze(-1)

    tr_loader = DataLoader(
        TensorDataset(X_tr, y_tr),
        batch_size=32,
        shuffle=False
    )

    v_loader = DataLoader(
        TensorDataset(X_v, y_v),
        batch_size=32,
        shuffle=False
    )

    return tr_loader, v_loader


sequence_experiment = []

for seq_len in [30, 60, 90]:
    tr_loader, v_loader = make_loaders_for_sequence_length(seq_len)

    model = LSTMModel(input_size=1, hidden_size=50, num_layers=1)

    _, train_loss, val_loss, _, exp_time = train_model(
        model,
        tr_loader,
        v_loader,
        epochs=10,
        learning_rate=0.001
    )

    sequence_experiment.append({
        "Sequence Length": seq_len,
        "Final Train Loss": train_loss[-1],
        "Final Validation Loss": val_loss[-1],
        "Training Time (sec)": exp_time
    })

sequence_results = pd.DataFrame(sequence_experiment)
sequence_results


In [ ]:
# Cell 30 - Hidden Size Experiment

hidden_experiment = []

for hidden_size in [32, 50, 100]:
    model = LSTMModel(
        input_size=1,
        hidden_size=hidden_size,
        num_layers=1
    )

    _, train_loss, val_loss, _, exp_time = train_model(
        model,
        train_loader,
        val_loader,
        epochs=10,
        learning_rate=0.001
    )

    hidden_experiment.append({
        "Hidden Size": hidden_size,
        "Parameters": sum(p.numel() for p in model.parameters()),
        "Final Train Loss": train_loss[-1],
        "Final Validation Loss": val_loss[-1],
        "Training Time (sec)": exp_time
    })

hidden_results = pd.DataFrame(hidden_experiment)
hidden_results


In [ ]:
# Cell 31 - Stacked LSTM Experiment

stacked_lstm = LSTMModel(
    input_size=1,
    hidden_size=50,
    num_layers=2
)

print(stacked_lstm)

print(
    "\nStacked LSTM parameter count:",
    sum(p.numel() for p in stacked_lstm.parameters())
)


In [ ]:
# Cell 32 - Train Stacked LSTM

stacked_lstm, stacked_train_losses, stacked_val_losses, stacked_gradients, stacked_time = train_model(
    stacked_lstm,
    train_loader,
    val_loader,
    epochs=10,
    learning_rate=0.001
)

stacked_results = evaluate_model(
    stacked_lstm,
    test_loader,
    scaler
)

print("\nStacked LSTM Test Results")
print("-------------------------")
print("RMSE:", stacked_results["RMSE"])
print("MAE :", stacked_results["MAE"])
print("R²  :", stacked_results["R2"])
print("Training time:", round(stacked_time, 2), "seconds")


In [ ]:
# Cell 33 - Best Model

best_row = results.loc[results["RMSE"].idxmin()]

print("Best model based on test RMSE:", best_row["Model"])
print("Best RMSE:", best_row["RMSE"])
print("Best MAE:", best_row["MAE"])
print("Best R²:", best_row["R2"])


In [ ]:
# Cell 34 - Final Prediction Table

# All three models use the same test targets.
final_predictions = pd.DataFrame({
    "Actual Price": rnn_results["Actual"],
    "Vanilla RNN": rnn_results["Predicted"],
    "LSTM": lstm_results["Predicted"],
    "GRU": gru_results["Predicted"]
})

print(final_predictions.head(20))


In [ ]:
# Cell 35 - Save Results

results.to_csv(
    "model_comparison_results.csv",
    index=False
)

sequence_results.to_csv(
    "sequence_length_experiment.csv",
    index=False
)

hidden_results.to_csv(
    "hidden_size_experiment.csv",
    index=False
)

final_predictions.to_csv(
    "final_predictions.csv",
    index=False
)

print("Result files saved successfully.")
